# GENA_LM vs AlphaGenome Ontology-Level Comparison

Average all cell IDs belonging to the same ontology, then compare GENA_LM and AlphaGenome with GT.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "downstream_tasks").exists():
    REPO = Path("/home/jovyan/dpanc/GENA_LM/GENA_LM_expression_branch")

BENCHMARK_ROOT = Path("/home/jovyan/dpanc/benchmarking")
DATA = BENCHMARK_ROOT / "data"
GENA_ROOT = BENCHMARK_ROOT / "GENA_LM"
ALPHAGENOME_ROOT = BENCHMARK_ROOT / "AlphaGenome"

sys.path.insert(0, str(REPO / "downstream_tasks/expression_prediction/gena_lm_benchmark/scripts"))
from score_ct_specificity import score_predictions

import json


In [ ]:
split = "test"       # valid or test
gena_model = "dev_loss"
ag_len_window = "1Mb"

GENA_PRED_PATH = GENA_ROOT / "predictions_results" / gena_model / f"gena_lm_{split}_json812_predictions.csv"
AG_PRED_PATH = ALPHAGENOME_ROOT / "predictions" / f"results-seq_len{ag_len_window}_json812" / f"alphagenome_supported_predictions_{split}_intervals.csv"
TRUE_PATH = DATA / "borzoi_all_ids_qnorm_matrix.csv"
ONTOLOGY_JSON = ALPHAGENOME_ROOT / "data/alphagenome_track_to_qnorm_ids_all.json"

print("GENA:", GENA_PRED_PATH)
print("AG:", AG_PRED_PATH)
print("GT:", TRUE_PATH)
print("ontology:", ONTOLOGY_JSON)


In [ ]:
gena_pred = pd.read_csv(GENA_PRED_PATH)
ag_pred = pd.read_csv(AG_PRED_PATH)
true_raw = pd.read_csv(TRUE_PATH)

with open(ONTOLOGY_JSON) as f:
    ontology_to_ids = json.load(f)

for df in [gena_pred, ag_pred]:
    if "gene_id" not in df.columns:
        df.rename(columns={df.columns[0]: "gene_id"}, inplace=True)

print("gena_pred:", gena_pred.shape)
print("ag_pred:", ag_pred.shape)
print("true_raw:", true_raw.shape)
print("ontologies:", len(ontology_to_ids))
display(gena_pred.head())
display(ag_pred.head())
display(true_raw.head())


In [ ]:
metadata_cols = ["gene_id", "id", "original_id", "targets_identifier_base", "strand_specificity"]
metadata_cols = [c for c in metadata_cols if c in true_raw.columns]
gene_cols = [c for c in true_raw.columns if c not in metadata_cols]
cell_id_col = "gene_id" if "gene_id" in true_raw.columns else "id"

true_gene_by_cell = true_raw.set_index(cell_id_col)[gene_cols].T.reset_index().rename(columns={"index": "gene_id"})

true_log = true_gene_by_cell.copy()
expr_cols = [c for c in true_log.columns if c != "gene_id"]
true_log[expr_cols] = np.log2(true_log[expr_cols].astype(float) + 1)

print("true_gene_by_cell:", true_gene_by_cell.shape)
print("true_log:", true_log.shape)
display(true_log.head())


In [ ]:
def mean_by_ontology(df, ontology_to_ids, name):
    df = df.set_index("gene_id")
    out = {}
    rows = []

    for ontology_name, cell_ids in ontology_to_ids.items():
        present_ids = [cell_id for cell_id in cell_ids if cell_id in df.columns]
        rows.append({"source": name, "ontology": ontology_name, "n_ids_in_ontology": len(cell_ids), "n_ids_present": len(present_ids)})
        if len(present_ids) == 0:
            continue
        out[ontology_name] = df[present_ids].astype(float).mean(axis=1)

    out = pd.DataFrame(out)
    out.insert(0, "gene_id", out.index)
    return out.reset_index(drop=True), pd.DataFrame(rows)


def corr_summary(true_df, pred_df):
    common_genes = true_df.index.intersection(pred_df.index)
    common_cells = true_df.columns.intersection(pred_df.columns)
    true = true_df.loc[common_genes, common_cells]
    pred = pred_df.loc[common_genes, common_cells]

    cell_corrs = []
    for cell in common_cells:
        true_vec = true[cell].astype(float).values
        pred_vec = pred[cell].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) >= 2 and np.std(true_vec) > 0 and np.std(pred_vec) > 0:
            cell_corrs.append(np.corrcoef(true_vec, pred_vec)[0, 1])

    gene_corrs = []
    for gene in common_genes:
        true_vec = true.loc[gene].astype(float).values
        pred_vec = pred.loc[gene].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) >= 4 and np.std(true_vec) > 0 and np.std(pred_vec) > 0:
            gene_corrs.append(np.corrcoef(true_vec, pred_vec)[0, 1])

    return {"corr_genes": float(np.mean(cell_corrs)) if cell_corrs else np.nan, "corr_cells": float(np.mean(gene_corrs)) if gene_corrs else np.nan, "n_cells": len(common_cells), "n_genes": len(common_genes)}


In [ ]:
gena_ontology, gena_summary = mean_by_ontology(gena_pred, ontology_to_ids, "GENA")
ag_ontology, ag_summary = mean_by_ontology(ag_pred, ontology_to_ids, "AG")
true_ontology, true_summary = mean_by_ontology(true_log, ontology_to_ids, "GT_log2p")

ontology_summary = pd.concat([gena_summary, ag_summary, true_summary], ignore_index=True)

print("gena_ontology:", gena_ontology.shape)
print("ag_ontology:", ag_ontology.shape)
print("true_ontology:", true_ontology.shape)
display(ontology_summary.head())
display(gena_ontology.head())


In [ ]:
gena_i = gena_ontology.set_index("gene_id")
ag_i = ag_ontology.set_index("gene_id")
true_i = true_ontology.set_index("gene_id")

common_genes = true_i.index.intersection(gena_i.index).intersection(ag_i.index)
common_ontologies = true_i.columns.intersection(gena_i.columns).intersection(ag_i.columns)

true_aligned = true_i.loc[common_genes, common_ontologies]
gena_aligned = gena_i.loc[common_genes, common_ontologies]
ag_aligned = ag_i.loc[common_genes, common_ontologies]

corr_result = pd.DataFrame([
    {"model": "GENA", **corr_summary(true_aligned, gena_aligned)},
    {"model": "AlphaGenome", **corr_summary(true_aligned, ag_aligned)},
])

print("common genes:", len(common_genes))
print("common ontologies:", len(common_ontologies))
display(corr_result)
